In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0', 'python-dotenv>=1.0.0',
    'pyyaml>=6.0', 'requests>=2.32.0',
], check=True)

In [ ]:
import os, json, time, threading
from pathlib import Path
from datetime import datetime
import yaml, requests
from huggingface_hub import HfApi, CommitOperationAdd

WORK_DIR        = Path('/kaggle/working')
EPISODES_PATH   = WORK_DIR / 'agent_episodes.jsonl'
DUMMY_ENV_PATH  = WORK_DIR / 'dummy_env.json'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p4c.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

MAX_RETRIES  = 12
COMMIT_DELAY = 3.0

In [ ]:
SECRETS = load_secrets(require_gemini=True)
HF_TOKEN    = SECRETS['HF_TOKEN_PRIMARY']
with open(CONFIG_DIR / 'hf_repos.yaml') as f: repos_cfg = yaml.safe_load(f)
STAGE3_REPO = repos_cfg['repos']['stage3_agent']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage3: {STAGE3_REPO}')

In [ ]:
if not EPISODES_PATH.exists():
    raise FileNotFoundError('agent_episodes.jsonl not found — run p4b first')

total_episodes = 0
domain_counts  = {}
diff_counts    = {}
outcome_counts = {}

with open(EPISODES_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        ep = json.loads(line)
        total_episodes += 1
        d = ep.get('domain','?')
        domain_counts[d]  = domain_counts.get(d, 0) + 1
        df = ep.get('difficulty','?')
        diff_counts[df]   = diff_counts.get(df, 0) + 1
        oc = ep.get('outcome','?')
        outcome_counts[oc] = outcome_counts.get(oc, 0) + 1

print(f'[p4c] {total_episodes} episodes')
print(f'  by domain    : {domain_counts}')
print(f'  by difficulty: {diff_counts}')
print(f'  by outcome   : {outcome_counts}')

uploads = [
    (str(EPISODES_PATH), 'agent_episodes.jsonl', f'{total_episodes} agent episodes'),
]
if DUMMY_ENV_PATH.exists():
    uploads.append((str(DUMMY_ENV_PATH), 'dummy_env.json', 'dummy customer profiles'))

for local_path, repo_path, label in uploads:
    print(f'\n[upload] {repo_path}...')
    for attempt in range(MAX_RETRIES):
        try:
            HF_API.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=repo_path,
                repo_id=STAGE3_REPO,
                repo_type='dataset',
                commit_message=label,
            )
            print(f'  uploaded: {repo_path}')
            break
        except Exception as e:
            wait = min(2**attempt, 120)
            print(f'  attempt {attempt+1}/{MAX_RETRIES} failed: {e} — retry in {wait}s')
            time.sleep(wait)

with open(CONFIG_DIR / 'tool_registry.yaml') as f:
    tool_registry = yaml.safe_load(f)
for attempt in range(MAX_RETRIES):
    try:
        HF_API.upload_file(
            path_or_fileobj=json.dumps(tool_registry, ensure_ascii=False, indent=2).encode(),
            path_in_repo='tool_registry.json',
            repo_id=STAGE3_REPO,
            repo_type='dataset',
            commit_message='tool_registry.json',
        )
        print('  uploaded: tool_registry.json')
        break
    except Exception as e:
        time.sleep(min(2**attempt, 120))

stats = {
    'total_episodes':  total_episodes,
    'by_domain':       domain_counts,
    'by_difficulty':   diff_counts,
    'by_outcome':      outcome_counts,
    'generated_at':    datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
}
for attempt in range(MAX_RETRIES):
    try:
        HF_API.upload_file(
            path_or_fileobj=json.dumps(stats, indent=2).encode(),
            path_in_repo='stats.json',
            repo_id=STAGE3_REPO,
            repo_type='dataset',
            commit_message='stats.json',
        )
        print('  uploaded: stats.json')
        break
    except Exception as e:
        time.sleep(min(2**attempt, 120))

print(f'\n[done] stage3 repo: https://huggingface.co/datasets/{STAGE3_REPO}')
print('[done] pipeline 4 complete — train stages 0-4, then run pipeline 5')